# Import and Evaluate a Hugging Face Dataset

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/huggingface-dataset-import)

Ridgeline Outfitters wants cancellation replies that fit in a chat window. You'll import three synthetic support exchanges from Hugging Face and flag any reply outside a 1 to 100 word limit. The result is a CSV with the customer instruction, the original reply, and the length check for each row.

## Set up one Python environment

Use a Python runtime with `venv`. The scripts below run in `.venv`, separate from your notebook kernel. Enter keys for the workspace where you want to create the dataset.

In [ ]:
%%bash -e
python3 -m venv .venv
source .venv/bin/activate
python -m pip install futureagi==0.7.0

In [ ]:
import os
from getpass import getpass

# Exported variables work in Jupyter; otherwise enter the keys at the hidden prompts.
for name in ['FI_API_KEY', 'FI_SECRET_KEY']:
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value:
            raise ValueError(f"Missing {name}")
        os.environ[name] = value
        del value

## Choose the rows to import

Import the first three exchanges from the public Bitext `train` split. Each run creates a new dataset. Keep its ID to export scores or resume an interrupted run.

In [ ]:
%%writefile import_replies.py
from uuid import uuid4

from fi.platform import Client
from fi.utils.types import ModelTypes

with Client() as client:
    created = client.datasets.create_from_huggingface(
        name=f"ridgeline-cancellations-{uuid4().hex[:8]}",
        model_type=ModelTypes.GENERATIVE_LLM.value,
        huggingface_dataset_name="bitext/Bitext-customer-support-llm-chatbot-training-dataset",
        huggingface_dataset_config="default",
        huggingface_dataset_split="train",
        num_rows=3,
    )
    print(created.dataset_id)

In [ ]:
import subprocess
from uuid import UUID

python = ".venv/bin/python"
result = subprocess.run([python, "import_replies.py"], check=True, stdout=subprocess.PIPE, text=True)
dataset_id = str(UUID(result.stdout.strip().splitlines()[-1]))
print("Created dataset:", dataset_id)

## Score the replies and save a CSV

Map the length check to `response`, not `instruction`. The script waits for the import, starts one evaluation, and exports three scored rows. If you interrupt it after the evaluation starts, use `--resume` with the existing dataset ID.

In [ ]:
%%writefile score_replies.py
import argparse
import csv
import time

from fi.platform import Client

parser = argparse.ArgumentParser()
parser.add_argument("dataset_id")
parser.add_argument("--resume", action="store_true", help="Wait and export without starting a run")
args = parser.parse_args()
dataset_id = args.dataset_id
check = "reply-length"

with Client() as client:
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        table = client.datasets.get(dataset_id, page_size=3)
        cols = {c.name: c.id for c in table.column_config}
        rows = [row.model_dump() for row in table.table or []]
        if {"instruction", "response"} <= cols.keys() and len(rows) == 3 and all(
            row.get(cols[field], {}).get("cell_value")
            for row in rows for field in ("instruction", "response")
        ):
            break
        time.sleep(2)
    else:
        raise TimeoutError(f"Expected three complete support exchanges in {dataset_id}")

    def attached_check():
        items = client.datasets.evals.list(dataset_id, eval_type="user").evals
        return next(item for item in items if item["name"] == check)

    if not args.resume:
        page = 0
        while True:
            batch = client.evals.templates.list(
                owner_filter="system", search="word_count_in_range", page=page, page_size=20,
            )
            match = next((t for t in batch.items if t.name == "word_count_in_range"), None)
            if match:
                template = client.evals.templates.get(match.id)
                break
            if not batch.items or (page + 1) * batch.page_size >= batch.total:
                raise LookupError("No system template named word_count_in_range")
            page += 1

        if set(template.required_keys) != {"text"}:
            raise ValueError(f"Unexpected inputs: {template.required_keys}")
        client.datasets.evals.add(
            dataset_id, name=check, template_id=template.id, run=False,
            config={"mapping": {"text": cols["response"]},
                    "params": {"min_words": 1, "max_words": 100}},
        )

        client.datasets.evals.run(dataset_id, user_eval_ids=[attached_check()["id"]])
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        state = attached_check()["status"]
        if state == "Completed":
            break
        if state not in {"NotStarted", "Queued", "Running"}:
            raise RuntimeError(f"{check}: {state}")
        time.sleep(2)
    else:
        raise TimeoutError(f"{check} still shows {state} on {dataset_id}")

    table = client.datasets.get(dataset_id, page_size=3)
    cols = {c.name: c.id for c in table.column_config}
    fields = ["instruction", "response", check]
    path = f"ridgeline-replies-{dataset_id}.csv"
    records = []
    for row in table.table or []:
        cells = row.model_dump()
        record = {}
        for field in fields:
            cell = cells.get(cols[field], {})
            value = cell.get("cell_value")
            if cell.get("status") == "error" or value is None or value == "":
                raise RuntimeError(f"Missing or failed {field} in row {row.row_id}")
            record[field] = value
        records.append(record)
    if len(records) != 3:
        raise RuntimeError(f"Expected 3 scored rows, got {len(records)}")
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)
    print(path)

In [ ]:
subprocess.run([python, "score_replies.py", dataset_id], check=True);

## Interpret the results

Read the CSV from the notebook directory. A reply over 100 words fails the length check; its length alone does not tell you whether its advice is correct.

In [ ]:
import csv

with open(f"ridgeline-replies-{dataset_id}.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(len(row["response"].split()), "words |", row["reply-length"], "|", row["instruction"])